# Learning-rate experiment

This notebook includes three training runs for the advanced model with three different **learning rates**:
- `1e-3`
- `1e-4`
- `1e-5`

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train import TrainConfig, run_training

## Configuration

We will use the same training parameters used in the previous experiments, as they proved adequate for this model. The only parameter changed between runs is the learning rate.

In [2]:
DATA_DIR = PROJECT_ROOT / 'data'
SPLITS_FILE = DATA_DIR / 'splits.json'
CHECKPOINT_ROOT = PROJECT_ROOT / 'results' / 'checkpoints'
LOG_ROOT = PROJECT_ROOT / 'results' / 'logs'

HARD_ATTACKS = [
    {'name': 'none'},
    {'name': 'gaussian_noise', 'std': 0.01},
    {'name': 'gaussian_noise', 'std': 0.03},
    {'name': 'gaussian_blur', 'kernel_size': 3, 'sigma': 0.5},
    {'name': 'downscale', 'scale_factor': 0.75},
    {'name': 'rotation', 'angle': -2.0},
    {'name': 'rotation', 'angle': 2.0},
    {'name': 'gaussian_noise', 'std': 0.05},
    {'name': 'gaussian_blur', 'kernel_size': 5, 'sigma': 1.0},
    {'name': 'downscale', 'scale_factor': 0.5},
    {'name': 'rotation', 'angle': -5.0},
    {'name': 'rotation', 'angle': 5.0},
]

LEARNING_RATES = (1e-3, 1e-4, 1e-5)
EPOCHS = 60
BATCH_SIZE = 16
NUM_WORKERS = 4

def create_config(learning_rate):
    rate_label = f'{learning_rate:.0e}'
    experiment_name = f'lr_experiment_{rate_label}'
    return TrainConfig(
        experiment_name=experiment_name,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        message_length=16,
        encoder_channels=40,
        decoder_channels=40,
        decoder_normalization='batch',
        encoder_max_delta=0.03,
        learning_rate=learning_rate,
        image_loss_weight=0.0,
        device='cuda',
        checkpoint_dir=str(CHECKPOINT_ROOT),
        log_path=str(LOG_ROOT / f'{experiment_name}.csv'),
        attack_configs=HARD_ATTACKS,
        validation_attack_configs=HARD_ATTACKS,
        print_every=1,
        checkpoint_metric='ber',
        architecture='advanced',
    )

configs = [create_config(rate) for rate in LEARNING_RATES]
pd.DataFrame([
    {
        'experiment_name': config.experiment_name,
        'learning_rate': config.learning_rate,
        'epochs': config.epochs,
        'batch_size': config.batch_size,
        'device': config.device,
        'num_workers': NUM_WORKERS,
    }
    for config in configs
])

,experiment_name,learning_rate,epochs,batch_size,device,num_workers
0,lr_experiment_1e-03,0.00100,60,16,cuda,4
1,lr_experiment_1e-04,0.00010,60,16,cuda,4
2,lr_experiment_1e-05,0.00001,60,16,cuda,4


## Training

We will run the training for all three learning rates and gather results for comparison.

In [ ]:

for config in configs:
    print(f'\nStarting {config.experiment_name} with learning rate {config.learning_rate:g}')
    _, _, history = run_training(data_dir=DATA_DIR, splits_file=SPLITS_FILE, config=config, num_workers=NUM_WORKERS, shuffle_train=True)


Starting lr_experiment_1e-03 with learning rate 0.001
Epoch 1/60 | train_loss=0.6813 | train_BER=0.4520 | train_exact=0.0000 | val_loss=0.6772 | val_BER=0.4462 | val_exact=0.0000 | val_PSNR=32.85 | 69.1s
Epoch 2/60 | train_loss=0.6685 | train_BER=0.4251 | train_exact=0.0000 | val_loss=0.6788 | val_BER=0.4385 | val_exact=0.0000 | val_PSNR=32.69 | 69.9s
Epoch 3/60 | train_loss=0.6648 | train_BER=0.4217 | train_exact=0.0000 | val_loss=0.6723 | val_BER=0.4369 | val_exact=0.0000 | val_PSNR=32.34 | 74.0s
Epoch 4/60 | train_loss=0.6569 | train_BER=0.4196 | train_exact=0.0000 | val_loss=0.6623 | val_BER=0.4351 | val_exact=0.0000 | val_PSNR=32.34 | 75.9s
Epoch 5/60 | train_loss=0.6449 | train_BER=0.4228 | train_exact=0.0000 | val_loss=0.6542 | val_BER=0.4264 | val_exact=0.0000 | val_PSNR=32.25 | 76.4s
Epoch 6/60 | train_loss=0.6334 | train_BER=0.4177 | train_exact=0.0003 | val_loss=0.6996 | val_BER=0.4449 | val_exact=0.0000 | val_PSNR=32.12 | 77.6s
Epoch 7/60 | train_loss=0.6224 | train_BER=0.